# 🚰 Step-by-Step: Membangun Data Pipeline Sederhana
Notebook ini diikuti bersama trainer selama sesi. Kita akan membangun pipeline:

```
API (JSON) ──► EXTRACT ──► TRANSFORM ──► LOAD ──► PostgreSQL ──► SQL ──► DASHBOARD
             requests      pandas       to_sql
```
Setiap bagian memiliki penjelasan, kode contoh, dan tugas kecil **🧪 Coba sendiri**.

## ⚙️ Persiapan
Pastikan kernel **.venv** sudah dipilih (pojok kanan atas) dan file `.env` sudah diisi, lalu jalankan sel-sel berikut.

In [ ]:
# ==== KONFIGURASI ====
# Kredensial database dibaca dari file .env di root repo (salin .env.example → .env, isi dari kartu akses).
import os
from dotenv import load_dotenv, find_dotenv
load_dotenv(find_dotenv(usecwd=True))

GITHUB_USER = "AisahAlfiyatusR"     # akun GitHub pemateri (repo asal yang Anda fork)
REPO        = "data-science-workshop"
RAW_BASE    = f"https://raw.githubusercontent.com/{GITHUB_USER}/{REPO}/main"
API_URL     = os.getenv("API_URL") or RAW_BASE + "/api/static/sales_page_{page}.json"

DB_HOST     = os.getenv("DB_HOST")
DB_PORT     = int(os.getenv("DB_PORT", 5432))
DB_NAME     = os.getenv("DB_NAME", "workshop")
DB_USER     = os.getenv("DB_USER")
DB_PASSWORD = os.getenv("DB_PASSWORD")
SCHEMA      = DB_USER   # setiap peserta punya schema pribadi bernama sama dengan username

if not (DB_HOST and DB_USER and DB_PASSWORD):
    print("⚠️ File .env belum ditemukan/diisi — bagian database tidak akan berjalan.")
else:
    print(f"Konfigurasi OK — login database sebagai {DB_USER} @ {DB_HOST}")

In [ ]:
import os, requests
import pandas as pd
from sqlalchemy import create_engine, text, Date
from sqlalchemy.engine import URL
pd.set_option("display.max_columns", 20)


def page_url(base, page, page_size=100):
    """URL satu halaman — mendukung API statis ('{page}' di URL) maupun API server (?page=)."""
    if "{page}" in base:
        return base.format(page=page)
    return f"{base}?page={page}&page_size={page_size}"

### Koneksi ke database
Kita memakai **SQLAlchemy** sebagai penghubung Python ↔ PostgreSQL. `URL.create` aman untuk password yang mengandung karakter khusus.

In [ ]:
engine = create_engine(URL.create("postgresql+psycopg2", username=DB_USER, password=DB_PASSWORD,
                                  host=DB_HOST, port=DB_PORT, database=DB_NAME),
                       pool_size=1, max_overflow=1, pool_pre_ping=True)   # hemat koneksi: ±70 peserta memakai server yang sama
with engine.connect() as conn:
    print(conn.execute(text("SELECT current_user, current_schema()")).fetchone())

---
# 1️⃣ EXTRACT — Mengambil data dari API
API mengembalikan data dalam **JSON** dan dibagi per **halaman** (pagination). Struktur respons:
```json
{"page": 1, "page_size": 100, "total": 304, "has_next": true, "data": [ {...}, {...} ]}
```
Mari lihat satu halaman dulu.

In [ ]:
resp = requests.get(page_url(API_URL, 1), timeout=30)
print("URL        :", resp.url)
print("Status code:", resp.status_code)
js = resp.json()
{k: (v[:2] if k == "data" else v) for k, v in js.items()}   # tampilkan 2 record pertama saja

Sekarang kita bungkus menjadi fungsi yang mengulang sampai `has_next = false`. Disediakan juga **fallback** ke CSV jika API bermasalah — di dunia kerja, pipeline harus siap menghadapi sumber data yang gagal.

In [ ]:
def extract_sales(api_url, page_size=100):
    """Mengambil seluruh halaman data dari API."""
    rows, page = [], 1
    while True:
        resp = requests.get(page_url(api_url, page, page_size), timeout=30)
        resp.raise_for_status()                      # error jika status bukan 2xx
        payload = resp.json()
        rows.extend(payload["data"])
        print(f"Halaman {page}: {len(payload['data'])} baris")
        if not payload.get("has_next"):
            break
        page += 1
    return pd.DataFrame(rows)


def load_csv_fallback(filename):
    """Cadangan jika API gagal: baca CSV dari folder data/ (clone lokal) atau langsung dari GitHub."""
    for p in (filename, f"data/{filename}", f"../data/{filename}"):
        if os.path.exists(p):
            return pd.read_csv(p, dtype=str, keep_default_na=False)
    url = f"{RAW_BASE}/data/{filename}"
    print("Membaca cadangan dari GitHub:", url)
    return pd.read_csv(url, dtype=str, keep_default_na=False)

In [ ]:
try:
    df_raw = extract_sales(API_URL)
except Exception as e:
    print("⚠️ API tidak dapat diakses:", e)
    df_raw = load_csv_fallback("sales_raw.csv")

print("Total baris:", len(df_raw))
df_raw.head()

### Kenali data mentah
Sebelum membersihkan, selalu **profiling** dulu.

In [ ]:
print("Dimensi:", df_raw.shape)
print("Duplikat:", df_raw.duplicated().sum())
print("Quantity kosong:", (df_raw["quantity"] == "").sum())
print("Kota unik:", sorted(df_raw["city"].unique()))
print("Contoh tanggal:", df_raw["transaction_date"].sample(5, random_state=1).tolist())
df_raw.dtypes

🧪 **Coba sendiri:** berapa transaksi dengan `unit_price` diawali tanda minus? *(petunjuk: `.str.startswith('-')`)*

In [ ]:
# jawaban Anda

---
# 2️⃣ TRANSFORM — Membersihkan data
Temuan profiling → aturan pembersihan:

| Masalah | Penanganan |
|---|---|
| Baris duplikat | `drop_duplicates()` |
| Format tanggal campuran (`YYYY-MM-DD` & `DD/MM/YYYY`) | parse dua format, gabungkan |
| Nama kota tidak konsisten | `strip()` + `title()` |
| Quantity kosong | isi 1 (asumsi bisnis — **selalu dokumentasikan asumsi!**) |
| Harga negatif | nilai absolut |
| Belum ada total | `total_amount = quantity × unit_price` |

In [ ]:
def transform_sales(df):
    df = df.copy()
    # 1) hapus baris duplikat persis
    df = df.drop_duplicates()
    # 2) tanggal: dua format berbeda -> parse masing-masing lalu gabungkan
    iso = pd.to_datetime(df["transaction_date"], format="%Y-%m-%d", errors="coerce")
    dmy = pd.to_datetime(df["transaction_date"], format="%d/%m/%Y", errors="coerce")
    df["transaction_date"] = iso.fillna(dmy).dt.date
    # 3) standarisasi nama kota
    df["city"] = df["city"].str.strip().str.title()
    # 4) quantity kosong -> 1 (asumsi minimal pembelian 1 unit)
    df["quantity"] = pd.to_numeric(df["quantity"], errors="coerce").fillna(1).astype(int)
    # 5) harga negatif -> nilai absolut (kesalahan input tanda)
    df["unit_price"] = pd.to_numeric(df["unit_price"], errors="coerce").abs()
    # kolom turunan
    df["total_amount"] = df["quantity"] * df["unit_price"]
    return df.reset_index(drop=True)

In [ ]:
df_clean = transform_sales(df_raw)
print("Sebelum:", len(df_raw), "| Sesudah:", len(df_clean))
df_clean.info()

### Validasi hasil
Jangan percaya begitu saja — **uji** hasil transform dengan `assert`.

In [ ]:
assert df_clean.duplicated().sum() == 0
assert df_clean["transaction_date"].notna().all()
assert (df_clean["unit_price"] >= 0).all()
assert df_clean["city"].nunique() == 5
print("✅ Data bersih valid:", sorted(df_clean["city"].unique()))

🧪 **Coba sendiri:** tampilkan total revenue per kategori menggunakan pandas `groupby`.

In [ ]:
# jawaban Anda

---
# 3️⃣ LOAD — Menyimpan ke PostgreSQL
`to_sql` membuat tabel otomatis. Parameter penting:
- `if_exists="replace"` → hapus & buat ulang (aman dijalankan berulang = **idempoten**)
- `if_exists="append"` → tambahkan baris (hati-hati duplikat bila dijalankan ulang)
- `dtype=` → memaksa tipe kolom tertentu, misalnya `DATE`

In [ ]:
df_clean.to_sql("sales_clean", engine, schema=SCHEMA, if_exists="replace", index=False,
                dtype={"transaction_date": Date()})

with engine.connect() as conn:
    n_db = conn.execute(text(f"SELECT COUNT(*) FROM {SCHEMA}.sales_clean")).scalar()
print(f"DataFrame: {len(df_clean)} baris | Database: {n_db} baris")
assert n_db == len(df_clean), "Jumlah baris tidak sama!"

In [ ]:
run_sql_preview = pd.read_sql(text(f"""
    SELECT column_name, data_type FROM information_schema.columns
    WHERE table_schema = '{SCHEMA}' AND table_name = 'sales_clean' ORDER BY ordinal_position"""), engine)
run_sql_preview

---
# 4️⃣ SQL — Analisis di database
Setelah data ada di database, analisis bisa dilakukan oleh siapa pun yang memahami SQL — termasuk tools BI.

In [ ]:
def run_sql(query):
    """Menjalankan query SELECT dan mengembalikan DataFrame."""
    with engine.connect() as conn:
        return pd.read_sql(text(query), conn)

In [ ]:
TABLE = f"{SCHEMA}.sales_clean"   # ganti ke "ref.sales_clean" jika Load Anda belum berhasil

In [ ]:
run_sql(f"""
    SELECT city, COUNT(*) AS transaksi, SUM(total_amount) AS revenue
    FROM {TABLE}
    GROUP BY city
    ORDER BY revenue DESC
""")

Window function untuk tren bulanan dan kontribusi:

In [ ]:
run_sql(f"""
    SELECT DATE_TRUNC('month', transaction_date)::date AS bulan,
           SUM(total_amount) AS revenue,
           ROUND(100.0 * SUM(total_amount) / SUM(SUM(total_amount)) OVER (), 1) AS pct_dari_total
    FROM {TABLE}
    GROUP BY 1 ORDER BY 1
""")

🧪 **Coba sendiri:** produk apa yang paling banyak terjual (quantity) di kota Malang?

In [ ]:
run_sql(f"""
    -- tulis query Anda
    SELECT 1
""")

---
# 5️⃣ DASHBOARD — Visualisasi
Dashboard yang baik: **KPI di atas**, tren di tengah, rincian di bawah.

In [ ]:
kpi = run_sql(f"""
    SELECT SUM(total_amount) AS revenue, COUNT(*) AS transaksi,
           COUNT(DISTINCT customer_id) AS pelanggan, AVG(total_amount) AS aov
    FROM {TABLE}""").iloc[0]
harian   = run_sql(f"SELECT transaction_date, SUM(total_amount) AS revenue FROM {TABLE} GROUP BY 1 ORDER BY 1")
per_kota = run_sql(f"SELECT city, SUM(total_amount) AS revenue FROM {TABLE} GROUP BY 1 ORDER BY 2 DESC")
per_kat  = run_sql(f"SELECT category, SUM(total_amount) AS revenue FROM {TABLE} GROUP BY 1")
top_prod = run_sql(f"""SELECT product_name AS produk, SUM(quantity) AS qty, SUM(total_amount) AS revenue
                       FROM {TABLE} GROUP BY 1 ORDER BY 3 DESC LIMIT 5""")

In [ ]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

rp = lambda x: f"Rp {x/1e6:,.1f} jt"
fig = make_subplots(
    rows=3, cols=4,
    specs=[[{"type": "indicator"}] * 4,
           [{"type": "xy", "colspan": 4}, None, None, None],
           [{"type": "xy", "colspan": 2}, None, {"type": "domain", "colspan": 2}, None]],
    row_heights=[0.18, 0.42, 0.40], vertical_spacing=0.10,
    subplot_titles=("", "", "", "", "Revenue Harian", "Revenue per Kota", "Komposisi Kategori"))

for i, (label, val, fmt) in enumerate([("Total Revenue", kpi.revenue / 1e6, ",.1f"),
                                       ("Transaksi", kpi.transaksi, ","),
                                       ("Pelanggan", kpi.pelanggan, ","),
                                       ("Rata-rata/Transaksi", kpi.aov / 1e3, ",.0f")], start=1):
    suffix = " jt" if i == 1 else (" rb" if i == 4 else "")
    fig.add_trace(go.Indicator(mode="number", value=float(val), title={"text": label},
                               number={"valueformat": fmt, "suffix": suffix,
                                       "prefix": "Rp " if i in (1, 4) else ""}), row=1, col=i)

fig.add_trace(go.Scatter(x=harian.transaction_date, y=harian.revenue, mode="lines",
                         line=dict(color="#0F766E"), name="Revenue"), row=2, col=1)
fig.add_trace(go.Bar(x=per_kota.city, y=per_kota.revenue, marker_color="#14B8A6",
                     text=[rp(v) for v in per_kota.revenue], textposition="outside"), row=3, col=1)
fig.add_trace(go.Pie(labels=per_kat.category, values=per_kat.revenue, hole=0.5,
                     marker=dict(colors=["#0F766E", "#F59E0B", "#94A3B8"])), row=3, col=3)
fig.update_layout(height=900, showlegend=False, title="Dashboard Penjualan Q1 2026",
                  template="plotly_white", margin=dict(t=80, l=40, r=40, b=40))
fig.show()

---
# 🎯 Rangkuman
| Tahap | Tools | Output |
|---|---|---|
| Extract | `requests` | `df_raw` (304 baris) |
| Transform | `pandas` | `df_clean` (300 baris) |
| Load | `SQLAlchemy` + `to_sql` | tabel `<schema>.sales_clean` |
| SQL | PostgreSQL | insight |
| Dashboard | Plotly / Looker Studio | visual |

Selanjutnya: kerjakan **Exercise 1–5**, lalu **Mini Project**.